# Tratamento do dataset de carros usados → `data/clean/dataset.csv`

Dataset: [Cars Dataset](https://www.kaggle.com/datasets/ayushparwal2026/cars-dataset) (`used_cars_data.csv`),
anúncios de carros usados em 11 cidades da Índia.

Este notebook **reproduz as decisões de tratamento da EDA** (`notebooks/eda.ipynb`, Aula 11),
sem refazer a análise, e exporta o resultado para `data/clean/dataset.csv`, que é o único
arquivo lido pelo `app.py`. Cada seção indica a parte da EDA de onde a decisão veio.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Funciona tanto rodando de dentro de notebooks/ quanto da raiz do projeto
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

df = pd.read_csv(RAIZ / "data/raw/used_cars_data.csv")
print(df.shape)
df.head()

## 1. Identificador (EDA, seção 1)

`S.No.` é só um identificador sequencial e não descreve o carro.

In [ ]:
df = df.drop(columns=["S.No."])

## 2. Marca e modelo (EDA, seção 1)

`Brand` é a primeira palavra de `Name` e `Model` é o restante; **Land Rover** é tratada à parte por ter duas palavras.

In [ ]:
def separar_marca(nome):
    texto = str(nome).strip()
    if texto.startswith("Land Rover "):
        return "Land Rover", texto[len("Land Rover "):]
    partes = texto.split(maxsplit=1)
    if len(partes) == 1:
        return partes[0], pd.NA
    return partes[0], partes[1]


df[["Brand", "Model"]] = df["Name"].apply(lambda x: pd.Series(separar_marca(x)))
df[["Name", "Brand", "Model"]].head()

## 3. Texto → número e valores inválidos (EDA, seção 2)

- Representações textuais de ausência (`null`) viram `NaN`.
- `Mileage`, `Engine` e `Power` viram `Mileage_value`, `Engine_cc` e `Power_bhp`.
- `Mileage_value <= 0` e `Seats <= 0` são inválidos e viram `NaN`.

In [ ]:
df = df.replace(["null", "NULL", "Null", "nan", "NaN", ""], np.nan)


def extrair_numero(coluna):
    return pd.to_numeric(
        coluna.astype("string").str.extract(r"([0-9]+(?:\.[0-9]+)?)")[0],
        errors="coerce",
    )


df["Mileage_value"] = extrair_numero(df["Mileage"])
df["Engine_cc"] = extrair_numero(df["Engine"])
df["Power_bhp"] = extrair_numero(df["Power"])

df.loc[df["Mileage_value"] <= 0, "Mileage_value"] = np.nan
df.loc[df["Seats"] <= 0, "Seats"] = np.nan

df[["Mileage_value", "Engine_cc", "Power_bhp", "Seats"]].isna().sum()

## 4. Idade do veículo (EDA, seção 3)

`Vehicle_Age` usa o maior ano da própria base como referência.

In [ ]:
ano_referencia = int(df["Year"].max())
df["Vehicle_Age"] = ano_referencia - df["Year"]
print("Ano de referência:", ano_referencia)

## 5. Ausentes e outliers (EDA, seção 4)

- **Outliers mantidos**: não há como afirmar que os valores extremos são erros.
- **Imputação pela mediana** em `Mileage_value`, `Engine_cc`, `Power_bhp` e `Seats` (poucos ausentes, distribuições assimétricas).
- **`Price` não é imputado**: as análises com preço usam só os anúncios com preço informado.
  Como o preço é a variável central do app, ficam apenas esses anúncios.
- `New_Price` (~86% ausente) não entra no conjunto principal e fica fora do app.

In [ ]:
for col in ["Mileage_value", "Engine_cc", "Power_bhp", "Seats"]:
    df[col] = df[col].fillna(df[col].median())

df = df.dropna(subset=["Price"])
df.shape

## 6. Colunas exportadas

As variáveis quantitativas são as mesmas da lista `VARS_QUANT` da EDA.
`Engine_cc` e `Seats` só têm valores inteiros, então são gravadas como inteiros.

In [ ]:
VARS_QUANT = ["Vehicle_Age", "Kilometers_Driven", "Engine_cc", "Power_bhp", "Mileage_value", "Seats", "Price"]
VARS_QUALI = ["Brand", "Model", "Location", "Fuel_Type", "Transmission", "Owner_Type"]

df[["Engine_cc", "Seats"]] = df[["Engine_cc", "Seats"]].astype(int)

df_clean = df[["Name", *VARS_QUALI, "Year", *VARS_QUANT]]
df_clean[VARS_QUANT].describe().round(2)

## 7. Exportar o dado tratado

In [ ]:
df_clean.to_csv(RAIZ / "data/clean/dataset.csv", index=False)
print("Exportado:", df_clean.shape)